# 04 — Dispatching rule

When a LITHOGRAPHY machine becomes free, which waiting lot does it pick?
The rule isn't in the data, so we infer it from observed decisions.

In [34]:
import numpy as np
import pandas as pd

from wafer_twin.data import load_tables, build_process

tables = load_tables()
process = build_process(tables)
litho = process[process["WORKSHOP"] == "LITHOGRAPHY"]

In [35]:
arr = tables["lot_arrival_table"]

# if every lot gets the same delay, due date order = fab arrival order
(arr["DUE_DATE"] - arr["ARRIVAL_DATE"]).value_counts()

61 days 00:00:00    9442
61 days 01:00:00    3662
62 days 00:00:00    3510
60 days 23:00:00    1656
58 days 23:00:00    1016
60 days 00:00:00     964
59 days 23:00:00     896
59 days 00:00:00     785
Name: count, dtype: int64

## Rebuilding each decision

For a machine starting a lot at time t, the queue it chose from is every lot that:
- has entered the LITHOGRAPHY queue before t and not started yet,
- has a recipe this machine can run (`interopt_table`).

Warm-up months are skipped and 20,000 decisions are sampled.

In [36]:
# Prepare arrays
lit = litho.sort_values("QUEUE_ENTRY").reset_index(drop=True)

q_entry = lit["QUEUE_ENTRY"].to_numpy()
begin = lit["PROCESS_BEGIN_DATE"].to_numpy()
arrival = lit["ARRIVAL_DATE"].to_numpy()
machines = lit["MACHINE"].to_numpy()
recipe_code, recipe_names = pd.factorize(lit["RECIPE"])
product_code, product_names = pd.factorize(lit["PRODUCT"])

# recipes each litho machine can run
io = tables["interopt_table"]
io = io[io["WORKSHOP"] == "LITHOGRAPHY"]
recipe_index = {r: i for i, r in enumerate(recipe_names)}
machine_recipes = {
    m: np.array([recipe_index[r] for r in g["RECIPE"] if r in recipe_index])
    for m, g in io.groupby("MACHINE")
}

# nobody waits longer than this, limits the search window
max_wait = (begin - q_entry).max()

In [37]:
#Add due dates to the arrays
lit = lit.merge(tables["lot_arrival_table"][["LOT", "DUE_DATE"]], on="LOT", how="left")
due = lit["DUE_DATE"].to_numpy()

In [38]:
# recipe of the lot the same machine ran just before
by_machine = lit.sort_values(["MACHINE", "PROCESS_BEGIN_DATE"])
lit["PREV_RECIPE"] = by_machine.groupby("MACHINE")["RECIPE"].shift(1)  # aligned on the index

# -1 when the machine had no previous lot
prev_code = lit["PREV_RECIPE"].map(recipe_index).fillna(-1).astype(int).to_numpy()

In [39]:
# Rebuild the sampled decisions
n_products, n_recipes = len(product_names), len(recipe_names)
rng = np.random.default_rng(0)
after_warmup = np.flatnonzero(begin >= np.datetime64("2023-04-01"))
sample = rng.choice(after_warmup, size=20_000, replace=False)

rows = []
for i in sample:
    t = begin[i]

    # only lots that entered the queue in the last max_wait can still be waiting
    lo = np.searchsorted(q_entry, t - max_wait, side="left")
    hi = np.searchsorted(q_entry, t, side="right")
    idx = np.arange(lo, hi)

    allowed = machine_recipes[machines[i]]
    compatible = np.isin(recipe_code[idx], allowed)
    waiting = idx[((begin[idx] > t) & compatible) | (idx == i)]

    if len(waiting) < 2:
        continue  # no real choice

    # waiting lots of the same product as the chosen one
    same = waiting[product_code[waiting] == product_code[i]]

    rows.append({
        "queue_size": len(waiting),
        "chosen_compatible": recipe_code[i] in allowed,
        "fifo_queue": q_entry[i] == q_entry[waiting].min(),
        "oldest_in_fab": arrival[i] == arrival[waiting].min(),
        "edd": due[i] == due[waiting].min(),
        "product": product_code[i],
        "product_counts": np.bincount(product_code[waiting], minlength=len(product_names)),
        "n_same_product": len(same),
        "fifo_within_product": q_entry[i] == q_entry[same].min(),
        "edd_within_product": due[i] == due[same].min(),
                "chosen_recipe": recipe_code[i],
        # lots in the queue for each (product, recipe) pair
        "pr_counts": np.bincount(
            product_code[waiting] * n_recipes + recipe_code[waiting],
            minlength=n_products * n_recipes,
        ),
        "prev_recipe": prev_code[i],
    })

decisions = pd.DataFrame(rows)
print(f"{len(decisions):,} decisions with a real choice")

19,462 decisions with a real choice


In [40]:
# Compare the simple rules
print("chosen recipe allowed on its machine:", f"{decisions['chosen_compatible'].mean():.1%}")
print("mean queue size:", round(decisions["queue_size"].mean(), 1))
print()
print(f"picked the oldest lot in the queue: {decisions['fifo_queue'].mean():.1%}")
print(f"picked the oldest lot in the fab:   {decisions['oldest_in_fab'].mean():.1%}")
print(f"expected by pure chance:            {(1 / decisions['queue_size']).mean():.1%}")

chosen recipe allowed on its machine: 100.0%
mean queue size: 16.0

picked the oldest lot in the queue: 8.8%
picked the oldest lot in the fab:   7.4%
expected by pure chance:            10.7%


In [41]:
# Products: chosen vs expected by chance
counts = np.vstack(decisions["product_counts"])

# if the pick were random, each product would win in proportion to its share of the queue
expected = (counts / counts.sum(axis=1, keepdims=True)).sum(axis=0)
chosen = np.bincount(decisions["product"], minlength=len(product_names))

pd.DataFrame(
    {"chosen": chosen, "expected_by_chance": expected.round(0), "ratio": (chosen / expected).round(2)},
    index=product_names,
).sort_index()

,chosen,expected_by_chance,ratio
PRODUCT_0001,5900,3759.0,1.57
PRODUCT_0002,2958,5891.0,0.50
PRODUCT_0003,6964,5610.0,1.24
PRODUCT_0004,3640,4202.0,0.87


In [42]:
# Due delay by product
arr = tables["lot_arrival_table"].copy()
arr["DELAY_DAYS"] = (arr["DUE_DATE"] - arr["ARRIVAL_DATE"]).dt.total_seconds() / 86400

arr.groupby("PRODUCT")["DELAY_DAYS"].agg(["min", "median", "max"]).round(2)

,min,median,max
PRODUCT,,,
PRODUCT_0001,58.96,61.0,62.0
PRODUCT_0002,58.96,61.0,62.0
PRODUCT_0003,58.96,61.0,62.0
PRODUCT_0004,58.96,61.0,62.0


In [43]:
# Rules within the chosen product

print(f"EDD over the whole queue: {decisions['edd'].mean():.1%}")

# only decisions where the machine had to pick between lots of the same product
multi = decisions[decisions["n_same_product"] >= 2]
print(f"\n{len(multi):,} decisions with 2+ lots of the chosen product")
print(f"FIFO within the product:  {multi['fifo_within_product'].mean():.1%}")
print(f"EDD within the product:   {multi['edd_within_product'].mean():.1%}")
print(f"expected by pure chance:  {(1 / multi['n_same_product']).mean():.1%}")

EDD over the whole queue: 7.4%

17,177 decisions with 2+ lots of the chosen product
FIFO within the product:  25.5%
EDD within the product:   25.2%
expected by pure chance:  24.8%


### Takeaways :

- Due delay is 59–62 days and the same for every product: due date = arrival + 2 calendar months
  (59 days for Jan–Feb, 62 for Jul–Aug; the −1 h cases come from daylight saving time)
- A machine only sees the lots its recipes allow: 16 lots on average, not the ~90 of the whole queue
- No age-based rule is used over the whole queue: FIFO 8.8%, oldest-in-fab 7.4%, earliest due date 7.4%,
  all below pure chance (10.7%)
- Products are clearly prioritised: *PRODUCT_0001* is picked 1.57× more than by chance,
  *PRODUCT_0003* 1.24×, *PRODUCT_0004* 0.87×, *PRODUCT_0002* only 0.50×
- Same order as the LITHOGRAPHY waiting times (42 h, 61 h, 80 h, 216 h per lot):
  product priority explains who waits most
- It also explains why age-based rules score below chance: *PRODUCT_0002* lots pile up and grow old,
  so the oldest lot in the queue is often one the machine rarely picks
- Within a product the pick looks random: FIFO 25.5%, EDD 25.2%, pure chance 24.8%
- Working hypothesis: pick a product according to a priority score, then any of its lots at random

## Priority score model

Each product gets a score s. A waiting lot is picked with probability
s(its product) / sum of s over all lots in the queue.

- Scores are relative, so PRODUCT_0001 is fixed at 1
- Fitted by maximum likelihood on 70% of the decisions, tested on the other 30%

In [44]:
# Train/test split
from scipy.optimize import minimize

counts = np.vstack(decisions["product_counts"])  # lots of each product in the queue
chosen = decisions["product"].to_numpy()          # product of the picked lot

rng = np.random.default_rng(1)
is_train = rng.random(len(decisions)) < 0.7
print(f"train: {is_train.sum():,}  test: {(~is_train).sum():,}")

train: 13,616  test: 5,846


In [45]:
# Fit the scores
ref = list(product_names).index("PRODUCT_0001")


def loglik(w, counts, chosen):
    # w = log-scores, log P(pick) = log s(chosen) - log(sum of s over the queue)
    return (w[chosen] - np.log(counts @ np.exp(w))).sum()


def neg_loglik(w_free, counts, chosen):
    w = np.insert(w_free, ref, 0.0)  # reference product stays at log-score 0
    return -loglik(w, counts, chosen)


res = minimize(
    neg_loglik,
    x0=np.zeros(len(product_names) - 1),
    args=(counts[is_train], chosen[is_train]),
    method="BFGS",
)
print("converged:", res.success)

w_hat = np.insert(res.x, ref, 0.0)
scores = pd.Series(np.exp(w_hat), index=product_names, name="score").sort_index()
scores.round(3)

converged: True


PRODUCT_0001    1.000
PRODUCT_0002    0.255
PRODUCT_0003    0.738
PRODUCT_0004    0.465
Name: score, dtype: float64

In [46]:
# Test on unseen decisions
test_counts, test_chosen = counts[~is_train], chosen[~is_train]
n_test = len(test_chosen)

ll_model = loglik(w_hat, test_counts, test_chosen)
ll_chance = loglik(np.zeros(len(product_names)), test_counts, test_chosen)  # all scores equal

# typical probability given to the lot that was actually picked
print(f"model:  {np.exp(ll_model / n_test):.2%}")
print(f"chance: {np.exp(ll_chance / n_test):.2%}")

model:  8.83%
chance: 8.01%


In [47]:
# Predicted vs actual picks per product (test set)
p = test_counts * np.exp(w_hat)
p = p / p.sum(axis=1, keepdims=True)  # prob of each product being picked, per decision

pd.DataFrame(
    {
        "actual": np.bincount(test_chosen, minlength=len(product_names)),
        "predicted": p.sum(axis=0).round(0),
    },
    index=product_names,
).sort_index()

,actual,predicted
PRODUCT_0001,1747,1783.0
PRODUCT_0002,899,900.0
PRODUCT_0003,2088,2064.0
PRODUCT_0004,1112,1098.0


### Takeaways :

- Scores relative to *PRODUCT_0001*: *PRODUCT_0003* 0.74, *PRODUCT_0004* 0.47, PRODUCT_0002 0.26
  (a *PRODUCT_0001* lot is ~4× more likely to be picked than a *PRODUCT_0002* lot)
- On unseen decisions, predicted picks per product match actual picks within 2%
- Probability given to the actual pick: 8.83% vs 8.01% by chance. The gain per decision is small because
  the pick is random within a product, but it is large over ~6,000 test decisions
- Decision: the twin uses this rule (pick a lot with probability proportional to its product score)

## Save the scores

The test is conclusive, so scores are refitted on all decisions and saved to `params/`.

In [48]:
# Refit on all decisions and save
from wafer_twin.params import PARAMS_DIR

# start from the train fit, should converge fast
res_all = minimize(neg_loglik, x0=res.x, args=(counts, chosen), method="BFGS")
print("converged:", res_all.success)

w_all = np.insert(res_all.x, ref, 0.0)
priority = pd.DataFrame(
    {"score": np.exp(w_all)},
    index=pd.Index(product_names, name="PRODUCT"),
).sort_index()

priority.to_csv(PARAMS_DIR / "litho_product_priority.csv")
priority.round(3)

converged: False


,score
PRODUCT,
PRODUCT_0001,1.000
PRODUCT_0002,0.257
PRODUCT_0003,0.747
PRODUCT_0004,0.471


## Adding a recipe effect

The first twin run showed that recipes few machines can run wait far too long in the twin
(RECIPE_0027: 3.6×, 5 machines). The real dispatching likely depends on the recipe too.

Three models compared on the same train/test split, weight of a lot = product part × recipe part:
- **product only**: the model above (baseline)
- **product + flexibility**: recipe part = (nb of machines able to run it)^(−β), one extra parameter
- **product + recipe scores**: one free score per recipe, 27 extra parameters

In [49]:
# how many litho machines can run each recipe
n_machines = np.array([
    sum(j in recs for recs in machine_recipes.values()) for j in range(n_recipes)
])

pr_counts = np.vstack(decisions["pr_counts"])
chosen_pr = decisions["product"].to_numpy() * n_recipes + decisions["chosen_recipe"].to_numpy()

# product and recipe of each (product, recipe) cell
prod_of = np.repeat(np.arange(n_products), n_recipes)
rec_of = np.tile(np.arange(n_recipes), n_products)

pd.Series(n_machines, index=recipe_names).sort_values()

RECIPE_0024     3
RECIPE_0021     5
RECIPE_0020     5
RECIPE_0027     5
RECIPE_0028     6
RECIPE_0003     6
RECIPE_0001     6
RECIPE_0004     6
RECIPE_0016     6
RECIPE_0025     7
RECIPE_0012     7
RECIPE_0009     7
RECIPE_0014     7
RECIPE_0010     7
RECIPE_0018     8
RECIPE_0006     8
RECIPE_0011     8
RECIPE_0029     8
RECIPE_0017     9
RECIPE_0015     9
RECIPE_0022     9
RECIPE_0008     9
RECIPE_0013     9
RECIPE_0023    10
RECIPE_0002    11
RECIPE_0026    12
RECIPE_0005    12
RECIPE_0030    12
dtype: int64

##### The three models

In [50]:
def loglik_cells(logw, counts, chosen):
    # logw = log-weight of each (product, recipe) cell
    return (logw[chosen] - np.log(counts @ np.exp(logw))).sum()


def product_part(theta):
    # PRODUCT_0001 fixed at 0, like before
    return np.insert(theta[: n_products - 1], ref, 0.0)[prod_of]


def logw_product(theta):
    return product_part(theta)


def logw_flex(theta):
    beta = theta[-1]
    return product_part(theta) - beta * np.log(n_machines[rec_of])


def logw_recipe(theta):
    # first recipe fixed at 0, the others relative to it
    v = np.insert(theta[n_products - 1 :], 0, 0.0)
    return product_part(theta) + v[rec_of]


models = {
    "product only": (logw_product, n_products - 1),
    "product + flexibility": (logw_flex, n_products),
    "product + recipe scores": (logw_recipe, n_products - 1 + n_recipes - 1),
}

In [51]:
# Fit and compare on the test set
train, test = is_train, ~is_train
results, fitted = {}, {}

for name, (build, n_params) in models.items():
    nll = lambda th, c, y: -loglik_cells(build(th), c, y)
    res = minimize(nll, np.zeros(n_params), args=(pr_counts[train], chosen_pr[train]), method="BFGS")
    fitted[name] = res.x

    ll_test = loglik_cells(build(res.x), pr_counts[test], chosen_pr[test])
    results[name] = {
        "params": n_params,
        "converged": res.success,
        "train_AIC": 2 * n_params + 2 * res.fun,
        "test_loglik": ll_test,
        "test_typical_prob": np.exp(ll_test / test.sum()),
    }

pd.DataFrame(results).T

,params,converged,train_AIC,test_loglik,test_typical_prob
product only,3,True,66267.072567,-14188.658578,0.088295
product + flexibility,4,True,65990.119932,-14125.025382,0.089261
product + recipe scores,30,False,65223.776915,-14007.836425,0.091069


In [52]:
# Flexibility exponent and recipe scores
print(f"beta = {fitted['product + flexibility'][-1]:.3f}")

v = np.insert(fitted["product + recipe scores"][n_products - 1 :], 0, 0.0)
pd.DataFrame(
    {"n_machines": n_machines, "recipe_score": np.exp(v - v.max())},
    index=recipe_names,
).sort_values("n_machines").round(3)

beta = 0.642


,n_machines,recipe_score
RECIPE_0024,3,0.405
RECIPE_0021,5,0.861
RECIPE_0020,5,0.429
RECIPE_0027,5,1.000
RECIPE_0028,6,0.648
RECIPE_0003,6,0.622
RECIPE_0001,6,0.432
RECIPE_0004,6,0.318
RECIPE_0016,6,0.694
RECIPE_0025,7,0.272


In [53]:
build, n_params = models["product + recipe scores"]
nll = lambda th, c, y: -loglik_cells(build(th), c, y)

# restart from where BFGS stopped
res_b = minimize(
    nll, fitted["product + recipe scores"],
    args=(pr_counts[train], chosen_pr[train]),
    method="L-BFGS-B", options={"maxiter": 5000},
)
print("converged:", res_b.success, "-", res_b.message)

ll_test_b = loglik_cells(build(res_b.x), pr_counts[test], chosen_pr[test])
print(f"test loglik: {ll_test_b:.1f}, typical prob: {np.exp(ll_test_b / test.sum()):.2%}")
fitted["product + recipe scores"] = res_b.x

converged: True - CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
test loglik: -14007.8, typical prob: 9.11%


### Takeaways :

- The recipe matters in real dispatching. Test log-likelihood (higher is better):
  product only −14,189, + flexibility −14,125 (1 extra param), + recipe scores −14,008 (27 extra params).
  The gain holds on unseen decisions, so it is not overfitting
- The recipe-score model first stopped before full convergence (BFGS). Restarted with L-BFGS-B,
  it converges with the same test result (−14,007.8, typical prob 9.11%), so the early stop didn't matter
- Flexibility exponent β = 0.64 > 0: recipes few machines can run are favoured
- Recipe scores go further: the highest are RECIPE_0027, 0021, 0016, 0028, 0012, 0003, i.e. heavily used
  recipes with only 5–7 machines, exactly those the twin v1 left waiting too long
- What seems to matter is load (much work for few machines) rather than flexibility alone:
  RECIPE_0024 has only 3 machines but little work and a mid score
- Decision: the twin v2 uses product scores × recipe scores

## Save the v2 scores

Refit on all decisions. Product and recipe scores go to two new files;
the v1 file is kept so the twin v1 can still be rerun.

In [54]:
res_all_b = minimize(
    nll, res_b.x, args=(pr_counts, chosen_pr),
    method="L-BFGS-B", options={"maxiter": 5000},
)
print("converged:", res_all_b.success, "-", res_all_b.message)

w_prod = np.insert(res_all_b.x[: n_products - 1], ref, 0.0)
v_rec = np.insert(res_all_b.x[n_products - 1 :], 0, 0.0)

prod_scores = pd.DataFrame(
    {"score": np.exp(w_prod)}, index=pd.Index(product_names, name="PRODUCT")
).sort_index()
# best recipe = 1
rec_scores = pd.DataFrame(
    {"score": np.exp(v_rec - v_rec.max())}, index=pd.Index(recipe_names, name="RECIPE")
).sort_index()

prod_scores.to_csv(PARAMS_DIR / "litho_priority_v2_product.csv")
rec_scores.to_csv(PARAMS_DIR / "litho_priority_v2_recipe.csv")
prod_scores.round(3)

converged: True - CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


,score
PRODUCT,
PRODUCT_0001,1.000
PRODUCT_0002,0.284
PRODUCT_0003,0.738
PRODUCT_0004,0.514


## Dynamic effects

Twin v2 still waits ~2× too long: static scores are not enough.
Two state-dependent effects are tested on top of v2:

- **same recipe**: lots with the recipe the machine just ran get weight × e^δ
  (changing recipe in lithography usually means changing the mask)
- **queue length**: each lot's weight × (lots of its recipe in the queue)^γ.
  With γ = 0 a recipe is already picked in proportion to its queue; γ > 0 means long queues are favoured even more

In [55]:
# Features for each decision
prev = decisions["prev_recipe"].to_numpy()
chosen_rec = decisions["chosen_recipe"].to_numpy()

# lots waiting for each recipe, per decision
rec_counts = pr_counts.reshape(len(decisions), n_products, n_recipes).sum(axis=1)

same = rec_of[None, :] == prev[:, None]                 # cell has the machine's previous recipe
log_nr = np.log(np.maximum(rec_counts[:, rec_of], 1))   # log queue length of the cell's recipe

In [56]:
rows = np.arange(len(prev))
prev_waiting = rec_counts[rows, np.maximum(prev, 0)]
has_prev = (prev >= 0) & (prev_waiting > 0)

stay = chosen_rec == prev
chance = prev_waiting / rec_counts.sum(axis=1)  # share of the queue with the previous recipe

print(f"{has_prev.sum():,} decisions where the previous recipe is waiting")
print(f"machine keeps the same recipe: {stay[has_prev].mean():.1%}")
print(f"expected by pure chance:       {chance[has_prev].mean():.1%}")

17,132 decisions where the previous recipe is waiting
machine keeps the same recipe: 76.7%
expected by pure chance:       37.2%


In [57]:
# Fit the four versions
n_base = n_products - 1 + n_recipes - 1
train_idx, test_idx = np.flatnonzero(is_train), np.flatnonzero(~is_train)


def loglik_dyn(theta, idx, use_same, use_queue):
    logw = logw_recipe(theta[:n_base])[None, :] + np.zeros((len(idx), 1))
    k = n_base
    if use_same:
        logw = logw + theta[k] * same[idx]
        k += 1
    if use_queue:
        logw = logw + theta[k] * log_nr[idx]

    num = logw[np.arange(len(idx)), chosen_pr[idx]]
    den = np.log((pr_counts[idx] * np.exp(logw)).sum(axis=1))
    return (num - den).sum()


variants = {
    "v2": (False, False),
    "v2 + same recipe": (True, False),
    "v2 + queue length": (False, True),
    "v2 + both": (True, True),
}

dyn_results, dyn_fitted = {}, {}
for name, (use_same, use_queue) in variants.items():
    # start from the v2 fit, new params at 0
    x0 = np.concatenate([res_b.x, np.zeros(use_same + use_queue)])
    res = minimize(
        lambda th: -loglik_dyn(th, train_idx, use_same, use_queue),
        x0, method="L-BFGS-B", options={"maxiter": 5000},
    )
    dyn_fitted[name] = res.x

    ll_test = loglik_dyn(res.x, test_idx, use_same, use_queue)
    dyn_results[name] = {
        "converged": res.success,
        "test_loglik": ll_test,
        "test_typical_prob": np.exp(ll_test / len(test_idx)),
        "delta": res.x[n_base] if use_same else np.nan,
        "gamma": res.x[-1] if use_queue else np.nan,
    }

pd.DataFrame(dyn_results).T

,converged,test_loglik,test_typical_prob,delta,gamma
v2,True,-14007.836426,0.091069,NaN,NaN
v2 + same recipe,True,-12178.20295,0.124535,2.200627,NaN
v2 + queue length,True,-13964.961281,0.091739,NaN,-0.276187
v2 + both,True,-12173.814515,0.124629,2.1743,-0.085964


### Takeaways :

- Machines keep their recipe: when the previous recipe is waiting, it is picked again 76.7% of the time
  vs 37.2% by pure chance
- Same-recipe effect: δ = 2.20, i.e. such a lot weighs ×9 (e^2.2). Test log-likelihood rises
  from −14,008 to −12,178 (+1,830), ten times the gain from recipe scores; typical prob 9.1% → 12.5%
- Consistent with lithography practice: changing recipe usually means changing the mask
- Queue length is rejected: γ is negative (−0.28 alone) and adds only +4 once the same-recipe effect
  is in the model (γ = −0.09). Alone, it was only catching part of the same-recipe effect
- Decision: twin v3 = product × recipe scores + same-recipe effect

## Save the v3 parameters

Refit v2 + same recipe on all decisions and save product scores, recipe scores
and the same-recipe factor (e^δ).

In [58]:
all_idx = np.arange(len(decisions))

res_v3 = minimize(
    lambda th: -loglik_dyn(th, all_idx, True, False),
    dyn_fitted["v2 + same recipe"], method="L-BFGS-B", options={"maxiter": 5000},
)
print("converged:", res_v3.success, "-", res_v3.message)

th = res_v3.x
w_prod = np.insert(th[: n_products - 1], ref, 0.0)
v_rec = np.insert(th[n_products - 1 : n_base], 0, 0.0)
delta = th[n_base]

pd.DataFrame(
    {"score": np.exp(w_prod)}, index=pd.Index(product_names, name="PRODUCT")
).sort_index().to_csv(PARAMS_DIR / "litho_priority_v3_product.csv")

pd.DataFrame(
    {"score": np.exp(v_rec - v_rec.max())}, index=pd.Index(recipe_names, name="RECIPE")
).sort_index().to_csv(PARAMS_DIR / "litho_priority_v3_recipe.csv")

extra = pd.DataFrame({"parameter": ["same_recipe_factor"], "value": [np.exp(delta)]})
extra.to_csv(PARAMS_DIR / "litho_priority_v3_extra.csv", index=False)

print(f"delta = {delta:.3f}, same-recipe factor = {np.exp(delta):.2f}")
print(pd.read_csv(PARAMS_DIR / "litho_priority_v3_product.csv").round(3))

converged: True - CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
delta = 2.190, same-recipe factor = 8.93
        PRODUCT  score
0  PRODUCT_0001  1.000
1  PRODUCT_0002  0.245
2  PRODUCT_0003  0.715
3  PRODUCT_0004  0.477
